# Validation
Checks that the key numbers in this project are correct and consistent across Python, SQL and Excel.

In [1]:
%run common.py
con = connect()

In [2]:
checks = {
    'Order items':     ("SELECT COUNT(*) FROM order_items", 180519),
    'Orders':          ("SELECT COUNT(*) FROM orders", 65752),
    'Customers':       ("SELECT COUNT(*) FROM customers", 20652),
    'Products':        ("SELECT COUNT(*) FROM products", 118),
    'Revenue':         ("SELECT ROUND(SUM(revenue)) FROM order_items", 33054402),
    'Profit':          ("SELECT ROUND(SUM(item_profit)) FROM order_items", 3966903),
    'Shipped orders':  ("SELECT COUNT(*) FROM orders WHERE is_shipped", 62897),
    'Late rate %':     ("SELECT ROUND(AVG(CASE WHEN is_late THEN 1 ELSE 0 END) * 100, 1) FROM orders WHERE is_shipped", 57.3),
    'Fraud orders':    ("SELECT COUNT(*) FROM orders WHERE is_fraud", 1488),
    'Profit 2017':     ("SELECT ROUND(SUM(item_profit)) FROM order_items WHERE order_year = 2017", 1304085),
    '10% cap (2017)':  ("SELECT ROUND(SUM(gross_sales * GREATEST(discount_rate - 0.10, 0))) FROM order_items WHERE order_year = 2017", 367409),
}

rows = []
for name, (query, expected) in checks.items():
    actual = con.sql(query).fetchone()[0]
    rows.append([name, actual, expected, abs(actual - expected) <= 1])

pd.DataFrame(rows, columns=['check', 'actual', 'expected', 'match'])

,check,actual,expected,match
0,Order items,"180,519.00","180,519.00",True
1,Orders,"65,752.00","65,752.00",True
2,Customers,"20,652.00","20,652.00",True
3,Products,118.00,118.00,True
4,Revenue,"33,054,402.00","33,054,402.00",True
5,Profit,"3,966,903.00","3,966,903.00",True
6,Shipped orders,"62,897.00","62,897.00",True
7,Late rate %,57.30,57.30,True
8,Fraud orders,"1,488.00","1,488.00",True
9,Profit 2017,"1,304,085.00","1,304,085.00",True


## 9.2 Same numbers in other tools
- **Excel pivot table:** revenue Grand Total = $33,054,402 ✓ (matches SQL)
- **Excel discount model:** 10% cap = $367,409 ✓ (matches SQL)
- **Power BI:** to check in Phase 10

## Export lookup tables for Power BI

In [3]:
from pathlib import Path
Path('../dashboard/data').mkdir(parents=True, exist_ok=True)

con.sql("SELECT * FROM products").df().to_csv('../dashboard/data/products.csv', index=False)
con.sql("SELECT * FROM customers").df().to_csv('../dashboard/data/customers.csv', index=False)
print('Saved')

Saved


In [4]:
print(pd.read_csv('../dashboard/data/products.csv').shape)
print(pd.read_csv('../dashboard/data/customers.csv').shape)

(118, 7)
(20652, 5)
